# **Predictions — Spaceship Titanic**

**Objetivo:** Generar predicciones sobre `test.csv` usando el mejor modelo entrenado.

| Artefacto | Origen |
|---|---|
| `models/best_advanced_model.pkl` | NB04.1 — modelo entrenado |
| `data/processed/train_features.csv` | NB03 — post-FE, pre-escalado (referencia de columnas + fit del scaler) |
| `data/raw/test.csv` | Dataset de competencia a predecir |

**Output:** `data/processed/submission.csv`


In [14]:
import sys
sys.path.insert(0, '../../')

MODEL_PATH      = '../../models/best_advanced_model.pkl'
TRAIN_FE_PATH   = '../../data/processed/train_features.csv'
TEST_RAW_PATH   = '../../data/raw/test.csv'
SUBMISSION_PATH = '../../data/processed/submission.csv'

NUMERIC_FEATURES = [
    'Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
    'GroupSize', 'CabinNumber', 'TotalSpending_Log', 'SpendingCategories',
]
CATEGORICAL_COLS = ['HomePlanet', 'Destination', 'Deck', 'AgeCategory']


In [15]:
import pandas as pd
import numpy as np
import joblib
import warnings
from sklearn.preprocessing import StandardScaler
from src.features.engineering import build_feature_set

warnings.filterwarnings('ignore')


In [16]:
model = joblib.load(MODEL_PATH)
print(f'Modelo cargado: {type(model).__name__}')


Modelo cargado: HistGradientBoostingClassifier


## **Preprocesamiento**

Mismo pipeline que NB03: feature engineering → label encoding → OHE → alineación de columnas → escalado.  
El scaler se ajusta sobre `train_features.csv` (exactamente como en NB03) y se aplica sobre test.


In [17]:
# Cargar referencias y datos a predecir
df_train_fe = pd.read_csv(TRAIN_FE_PATH)
df_test     = pd.read_csv(TEST_RAW_PATH)
test_ids    = df_test['PassengerId'].copy()

feature_cols = df_train_fe.drop(columns=['Transported']).columns.tolist()

# Feature engineering
df_test_fe = build_feature_set(df_test)

# Label encoding
df_test_fe['CryoSleep_Encoded'] = df_test_fe['CryoSleep'].map({True: 1, False: 0, 'Unknown': -1})
df_test_fe['Side_Encoded']      = df_test_fe['Side'].map({'P': 0, 'S': 1, 'Unknown': -1})

# OHE
df_test_enc = pd.get_dummies(df_test_fe, columns=CATEGORICAL_COLS, drop_first=False)

# Alinear columnas (cubre categorías presentes en train pero no en test)
X_test = df_test_enc.reindex(columns=feature_cols, fill_value=0)

# Escalado: fit en train, transform en test
scaler = StandardScaler()
scaler.fit(df_train_fe[NUMERIC_FEATURES])
X_test[NUMERIC_FEATURES] = scaler.transform(X_test[NUMERIC_FEATURES])

print(f'Test procesado: {X_test.shape}')


Test procesado: (4186, 35)


In [18]:
predictions = model.predict(X_test).astype(bool)

submission = pd.DataFrame({'PassengerId': test_ids, 'Transported': predictions})
submission.to_csv(SUBMISSION_PATH, index=False)

print(f'Registros  : {len(submission):,}')
print(f'Guardado en: {SUBMISSION_PATH}')
print(submission['Transported'].value_counts().to_string())


ValueError: array length 4186 does not match index length 4277